# 03 — Regime Detection & Ensemble

This notebook demonstrates the three-signal recession detection ensemble:

1. **Multivariate RSM** — Hamilton (1989) Markov-switching model on all 4 DFM
   factors simultaneously (2 regimes, 3 random restarts, probability clipping)
2. **Recession Probit** — Supervised probit model trained on NBER dates with
   yield curve lags (T10Y2Y, BAA10Y at 3- and 6-month leads)
3. **CFNAI threshold** — Chicago Fed index mapped through a normal CDF

**Ensemble weights:** RSM 25% · Probit 50% · CFNAI 25%


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import norm as sp_norm
%matplotlib inline


In [ ]:
# Fetch data and fit DFM (same as notebook 02)
from src.data.fred_client import FREDClient
from src.data.data_pipeline import DataPipeline
from src.models.dynamic_factor_model import DynamicFactorModel

client = FREDClient()
pipeline = DataPipeline(fred_client=client, start_date='1979-01-01')
panel = pipeline.run(save_vintage=False)

dfm = DynamicFactorModel(
    n_factors=4,
    factor_names=['real_activity', 'labor_market', 'inflation', 'financial_conditions'],
    max_iter=100,
)
dfm.fit(panel.dropna(how='all'))
factors = dfm.factors_
print(f'Factors: {factors.shape}')


In [ ]:
# --- Signal 1: Multivariate Regime-Switching Model ---
from src.models.regime_switching import RegimeSwitchingModel

rsm = RegimeSwitchingModel(
    n_regimes=2,
    regime_labels=['expansion', 'recession'],
    multivariate=True,
    n_restarts=3,
    max_iter=200,
)
rsm.fit(factors)

rsm_probs = rsm.get_recession_probability()
print(f'RSM recession probability (latest): {rsm_probs.iloc[-1]:.1%}')


In [ ]:
# --- Signal 2: Recession Probit (trained on NBER dates) ---
from src.models.recession_probit import RecessionProbit
from src.models.regime_backtest import NBER_RECESSIONS

# Build NBER binary series
nber = pd.Series(0, index=factors.index, dtype=float)
for start, end in NBER_RECESSIONS:
    mask = (nber.index >= pd.Timestamp(start)) & (nber.index <= pd.Timestamp(end))
    nber.loc[mask] = 1.0

# Build probit features: DFM factors + yield curve with leading lags
probit_feats = factors.copy()
for code in ['CFNAI', 'T10Y2Y', 'BAA10Y']:
    if code in panel.columns:
        probit_feats[code] = panel[code].reindex(factors.index).ffill()
for code in ['T10Y2Y', 'BAA10Y']:
    if code in panel.columns:
        s = panel[code].reindex(factors.index).ffill()
        for lag in [3, 6]:
            probit_feats[f'{code}_lag{lag}'] = s.shift(lag)
probit_feats = probit_feats.ffill()

# Train probit (use data up to 2020 to avoid COVID distortion)
train_idx = probit_feats.index[probit_feats.index <= '2020-04-30']
nber_train = nber.loc[train_idx]

probit = RecessionProbit(add_lags=3, regularization=1.0)
probit.fit(probit_feats.loc[train_idx], nber_train)
probit_proba = pd.Series(probit.predict_proba(probit_feats), index=probit_feats.index)

print(f'Probit recession probability (latest): {probit_proba.iloc[-1]:.1%}')


In [ ]:
# --- Signal 3: CFNAI threshold ---
cfnai = panel['CFNAI'].reindex(factors.index).ffill()
p_cfnai = cfnai.apply(lambda x: sp_norm.cdf(-x) if pd.notna(x) else 0.5)

print(f'CFNAI recession probability (latest): {p_cfnai.iloc[-1]:.1%}')


In [ ]:
# --- Ensemble: weighted combination ---
W_RSM, W_PROBIT, W_CFNAI = 0.25, 0.50, 0.25

ensemble = (
    W_RSM * rsm_probs.reindex(factors.index).fillna(0.5)
    + W_PROBIT * probit_proba.reindex(factors.index).fillna(0.5)
    + W_CFNAI * p_cfnai.reindex(factors.index).fillna(0.5)
)

print(f'\nEnsemble recession probability (latest): {ensemble.iloc[-1]:.1%}')
print(f'  RSM:    {rsm_probs.iloc[-1]:.1%} × {W_RSM}')
print(f'  Probit: {probit_proba.iloc[-1]:.1%} × {W_PROBIT}')
print(f'  CFNAI:  {p_cfnai.iloc[-1]:.1%} × {W_CFNAI}')


In [ ]:
# --- Visualise all three signals + ensemble vs NBER ---
nber_shade = [
    ('1990-07-01', '1991-03-01'),
    ('2001-03-01', '2001-11-01'),
    ('2007-12-01', '2009-06-01'),
    ('2020-02-01', '2020-04-01'),
]

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# RSM
axes[0].plot(rsm_probs.index, rsm_probs, color='#e74c3c', linewidth=1.2)
axes[0].set_ylabel('P(recession)')
axes[0].set_title('Signal 1: Multivariate RSM (weight = 25%)')
axes[0].set_ylim(0, 1)

# Probit
axes[1].plot(probit_proba.index, probit_proba, color='#8e44ad', linewidth=1.2)
axes[1].set_ylabel('P(recession)')
axes[1].set_title('Signal 2: Recession Probit (weight = 50%)')
axes[1].set_ylim(0, 1)

# CFNAI
axes[2].plot(p_cfnai.dropna().index, p_cfnai.dropna(), color='#2980b9', linewidth=1.2)
axes[2].set_ylabel('P(recession)')
axes[2].set_title('Signal 3: CFNAI Threshold (weight = 25%)')
axes[2].set_ylim(0, 1)

# Ensemble
axes[3].plot(ensemble.index, ensemble, color='#2c3e50', linewidth=2)
axes[3].axhline(0.5, color='grey', linewidth=0.8, linestyle='--', alpha=0.6)
axes[3].set_ylabel('P(recession)')
axes[3].set_title('Ensemble Recession Probability')
axes[3].set_ylim(0, 1)

for ax in axes:
    for start, end in nber_shade:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.2, color='red')

plt.tight_layout()
plt.show()
